# Silver Layer — Cleaned and Joined Tables
**Catalog:** `manufacturing_facility_ops`  
**Schema:** `silver`  
**Source:** `manufacturing_facility_ops.bronze.*`  
**Description:** Cleans bronze tables, casts data types, enriches with derived fields, and creates joined views that agents query. Produces 5 silver tables.

## 0. Setup

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark.sql("CREATE SCHEMA IF NOT EXISTS manufacturing_facility_ops.silver")

CATALOG = "manufacturing_facility_ops"
BRONZE  = "bronze"
SILVER  = "silver"

def write_silver(df, table_name):
    full_table = f"{CATALOG}.{SILVER}.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )
    count = spark.table(full_table).count()
    print(f"✅  {full_table}  —  {count} rows")

print("Schema ready:", f"{CATALOG}.{SILVER}")

Schema ready: manufacturing_facility_ops.silver


## 1. Silver Table — machine_status
Cleans machines table. Casts dates, adds days_since_last_maintenance, joins maintenance history for incident counts.

In [16]:
machines = spark.table(f"{CATALOG}.{BRONZE}.machines") \
    .withColumn("install_date",          F.to_date("install_date")) \
    .withColumn("last_maintenance",      F.to_date("last_maintenance")) \
    .withColumn("next_scheduled_maint",  F.to_date("next_scheduled_maint")) \
    .withColumn("days_since_last_maint", F.datediff(F.current_date(), F.col("last_maintenance")))

maint_history = spark.table(f"{CATALOG}.{BRONZE}.maintenance_history") \
    .withColumn("failure_date", F.to_date("failure_date"))

incident_counts = maint_history.groupBy("machine_id") \
    .agg(
        F.count("*").alias("total_incidents"),
        F.sum(F.when(
            F.col("failure_type").isin("Spindle Vibration", "Spindle Failure"), 1
        ).otherwise(0)).alias("spindle_incidents"),
        F.sum("downtime_hrs").alias("total_downtime_hrs"),
        F.sum("cost_usd").alias("total_maintenance_cost_usd")
    )

machine_status = machines.join(incident_counts, on="machine_id", how="left") \
    .fillna(0, subset=["total_incidents", "spindle_incidents", "total_downtime_hrs", "total_maintenance_cost_usd"])

write_silver(machine_status, "machine_status")
spark.table(f"{CATALOG}.{SILVER}.machine_status").show(8, truncate=False)

✅  manufacturing_facility_ops.silver.machine_status  —  8 rows


+----------+---------------+---------------+------------+------------+----------------+--------------------+--------+---------------------+---------------+-----------------+------------------+--------------------------+
|machine_id|machine_name   |production_line|status      |install_date|last_maintenance|next_scheduled_maint|location|days_since_last_maint|total_incidents|spindle_incidents|total_downtime_hrs|total_maintenance_cost_usd|
+----------+---------------+---------------+------------+------------+----------------+--------------------+--------+---------------------+---------------+-----------------+------------------+--------------------------+
|M-101     |CNC Lathe 101  |Line-1         |Operational |2019-03-12  |2025-04-01      |2025-07-01          |Bay A   |413                  |2              |0                |8.5               |1180                      |
|M-102     |CNC Lathe 102  |Line-3         |Under Review|2019-03-15  |2024-11-09      |2025-02-09          |Bay A   |556

## 2. Silver Table — active_work_orders
Filters to non-completed orders. Adds units_remaining, completion_pct, days_until_due. Joins machine status to flag at-risk machines.

In [17]:
work_orders = spark.table(f"{CATALOG}.{BRONZE}.work_orders") \
    .withColumn("due_date",           F.to_date("due_date")) \
    .withColumn("quantity_ordered",   F.col("quantity_ordered").cast(IntegerType())) \
    .withColumn("quantity_completed", F.col("quantity_completed").cast(IntegerType())) \
    .withColumn("defect_rate_pct",    F.col("defect_rate_pct").cast(DoubleType()))

active_orders = work_orders.filter(F.col("status") != "Completed") \
    .withColumn("units_remaining",  F.col("quantity_ordered") - F.col("quantity_completed")) \
    .withColumn("completion_pct",   F.round((F.col("quantity_completed") / F.col("quantity_ordered")) * 100, 1)) \
    .withColumn("days_until_due",   F.datediff(F.col("due_date"), F.current_date()))

machine_flags = spark.table(f"{CATALOG}.{SILVER}.machine_status") \
    .select("machine_id", "status", "spindle_incidents") \
    .withColumnRenamed("status", "machine_status")

active_work_orders = active_orders.join(machine_flags, on="machine_id", how="left") \
    .withColumn("machine_at_risk",
        F.col("machine_status").isin("Under Review", "Maintenance")
    )

write_silver(active_work_orders, "active_work_orders")
spark.table(f"{CATALOG}.{SILVER}.active_work_orders").show(10, truncate=False)

✅  manufacturing_facility_ops.silver.active_work_orders  —  7 rows


+----------+--------+---------------+-------------+----------------+------------------+---------------+----------+--------+-----------+---------------+---------------+--------------+--------------+--------------+-----------------+---------------+
|machine_id|order_id|product        |product_model|quantity_ordered|quantity_completed|production_line|due_date  |priority|status     |defect_rate_pct|units_remaining|completion_pct|days_until_due|machine_status|spindle_incidents|machine_at_risk|
+----------+--------+---------------+-------------+----------------+------------------+---------------+----------+--------+-----------+---------------+---------------+--------------+--------------+--------------+-----------------+---------------+
|M-103     |ORD-8804|Washing Machine|WM-7700      |200             |165               |Line-2         |2025-05-17|Standard|In Progress|2.1            |35             |82.5          |-367          |Operational   |0                |false          |
|M-102     |

## 3. Silver Table — maintenance_summary
Cleans maintenance history. Casts types, adds months_ago field, flags recurring failure types per machine.

In [18]:
maint_history = spark.table(f"{CATALOG}.{BRONZE}.maintenance_history") \
    .withColumn("failure_date", F.to_date("failure_date")) \
    .withColumn("downtime_hrs", F.col("downtime_hrs").cast(DoubleType())) \
    .withColumn("cost_usd",     F.col("cost_usd").cast(DoubleType())) \
    .withColumn("months_ago",   F.round(F.datediff(F.current_date(), F.col("failure_date")) / 30, 1))

failure_freq = maint_history.groupBy("machine_id", "failure_type") \
    .agg(F.count("*").alias("failure_type_count"))

maintenance_summary = maint_history \
    .join(failure_freq, on=["machine_id", "failure_type"], how="left") \
    .withColumn("is_recurring", F.col("failure_type_count") > 1) \
    .orderBy("machine_id", "failure_date")

write_silver(maintenance_summary, "maintenance_summary")
spark.table(f"{CATALOG}.{SILVER}.maintenance_summary").show(10, truncate=False)

✅  manufacturing_facility_ops.silver.maintenance_summary  —  10 rows


+----------+-----------------+-------------+------------+---------------------------------------------+----------+----------------------------------------+------------+--------+----------+------------------+------------+
|machine_id|failure_type     |work_order_id|failure_date|description                                  |technician|resolution                              |downtime_hrs|cost_usd|months_ago|failure_type_count|is_recurring|
+----------+-----------------+-------------+------------+---------------------------------------------+----------+----------------------------------------+------------+--------+----------+------------------+------------+
|M-101     |Coolant Leak     |WO-4310      |2024-08-14  |Coolant line fracture at joint B7            |R. Patel  |Replaced coolant line, pressure tested  |4.5         |620.0   |21.4      |1                 |false       |
|M-101     |Coolant Pump     |WO-4459      |2025-04-02  |Coolant pump output reduced 40%              |S. Kumar  |Re

## 4. Silver Table — inventory_availability
Cleans inventory. Adds availability_status flag and total stock value.

In [19]:
inventory = spark.table(f"{CATALOG}.{BRONZE}.inventory") \
    .withColumn("qty_available",  F.col("qty_available").cast(IntegerType())) \
    .withColumn("unit_cost_usd",  F.col("unit_cost_usd").cast(DoubleType())) \
    .withColumn("availability_status",
        F.when(F.col("qty_available") == 0, "Out of Stock")
         .when(F.col("qty_available") <= 2, "Low Stock")
         .otherwise("Available")
    ) \
    .withColumn("total_stock_value_usd",
        F.round(F.col("qty_available") * F.col("unit_cost_usd"), 2)
    )

write_silver(inventory, "inventory_availability")
spark.table(f"{CATALOG}.{SILVER}.inventory_availability").show(10, truncate=False)

✅  manufacturing_facility_ops.silver.inventory_availability  —  10 rows


+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+-------------------+---------------------+
|part_number|part_name            |machine_compatibility|warehouse  |bin_location|qty_available|unit_cost_usd|lead_time_days|availability_status|total_stock_value_usd|
+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+-------------------+---------------------+
|SP-4471    |CNC Spindle Assembly |M-101,M-102,M-105    |Warehouse B|B-12-04     |3            |1850.0       |Same Day      |Available          |5550.0               |
|SP-4472    |Drive Belt Primary   |M-103,M-104          |Warehouse A|A-05-11     |8            |120.0        |Same Day      |Available          |960.0                |
|SP-4490    |Position Encoder Unit|M-104,M-107          |Warehouse A|A-07-03     |2            |530.0        |Same Day      |Low Stock          |1060.0         

## 5. Silver Table — production_lines
Cleans production lines table. Casts numeric columns. Used by gold layer for rerouting capacity — keeps gold from reading bronze directly.

In [20]:
prod_lines = spark.table(f"{CATALOG}.{BRONZE}.production_lines") \
    .withColumn("max_capacity_units_day",      F.col("max_capacity_units_day").cast(IntegerType())) \
    .withColumn("current_load_units_day",       F.col("current_load_units_day").cast(IntegerType())) \
    .withColumn("available_capacity_units_day", F.col("available_capacity_units_day").cast(IntegerType())) \
    .withColumn("shift_hours",                  F.col("shift_hours").cast(IntegerType()))

write_silver(prod_lines, "production_lines")
spark.table(f"{CATALOG}.{SILVER}.production_lines").show(truncate=False)

✅  manufacturing_facility_ops.silver.production_lines  —  6 rows


+-------+---------------+-----------------------+----------------------+----------------------+----------------------------+-----------+-----------+
|line_id|line_name      |compatible_products    |max_capacity_units_day|current_load_units_day|available_capacity_units_day|shift_hours|status     |
+-------+---------------+-----------------------+----------------------+----------------------+----------------------------+-----------+-----------+
|Line-1 |Assembly Line 1|WM-7700,WM-8800        |180                   |140                   |40                          |16         |Operational|
|Line-2 |Assembly Line 2|WM-7700,CP-110         |200                   |175                   |25                          |16         |Operational|
|Line-3 |Assembly Line 3|WM-8800,WM-9000        |220                   |198                   |22                          |16         |Degraded   |
|Line-4 |Assembly Line 4|DU-3300,DU-4400        |160                   |95                    |65         

## 6. Validation — Silver Layer Summary

In [21]:
print(f"{'TABLE':<50} {'ROWS':>6} {'COLS':>6}")
print("-" * 65)
for t in ["machine_status", "active_work_orders", "maintenance_summary",
          "inventory_availability", "production_lines"]:
    full = f"{CATALOG}.{SILVER}.{t}"
    df   = spark.table(full)
    print(f"{full:<50} {df.count():>6} {len(df.columns):>6}")

print("\n--- KEY CHECKS ---")

print("\nM-102 summary:")
spark.table(f"{CATALOG}.{SILVER}.machine_status") \
    .filter(F.col("machine_id") == "M-102") \
    .select("machine_id", "status", "spindle_incidents", "total_downtime_hrs", "total_maintenance_cost_usd") \
    .show(truncate=False)

print("At-risk orders:")
spark.table(f"{CATALOG}.{SILVER}.active_work_orders") \
    .filter(F.col("machine_at_risk") == True) \
    .select("order_id", "product_model", "units_remaining", "due_date", "days_until_due", "status", "defect_rate_pct") \
    .show(truncate=False)

print("Production lines capacity:")
spark.table(f"{CATALOG}.{SILVER}.production_lines") \
    .select("line_id", "compatible_products", "max_capacity_units_day", "current_load_units_day", "available_capacity_units_day", "status") \
    .show(truncate=False)

TABLE                                                ROWS   COLS
-----------------------------------------------------------------


manufacturing_facility_ops.silver.machine_status        8     13


manufacturing_facility_ops.silver.active_work_orders      7     17


manufacturing_facility_ops.silver.maintenance_summary     10     12


manufacturing_facility_ops.silver.inventory_availability     10     10


manufacturing_facility_ops.silver.production_lines      6      8

--- KEY CHECKS ---

M-102 summary:


+----------+------------+-----------------+------------------+--------------------------+
|machine_id|status      |spindle_incidents|total_downtime_hrs|total_maintenance_cost_usd|
+----------+------------+-----------------+------------------+--------------------------+
|M-102     |Under Review|3                |28.0              |5630                      |
+----------+------------+-----------------+------------------+--------------------------+

At-risk orders:


+--------+-------------+---------------+----------+--------------+-----------+---------------+
|order_id|product_model|units_remaining|due_date  |days_until_due|status     |defect_rate_pct|
+--------+-------------+---------------+----------+--------------+-----------+---------------+
|ORD-8810|WM-8800      |88             |2025-05-18|-366          |In Progress|6.4            |
|ORD-8812|WM-8800      |152            |2025-05-19|-365          |At Risk    |8.3            |
|ORD-8819|WM-9000      |146            |2025-05-20|-364          |At Risk    |7.9            |
+--------+-------------+---------------+----------+--------------+-----------+---------------+

Production lines capacity:


+-------+-----------------------+----------------------+----------------------+----------------------------+-----------+
|line_id|compatible_products    |max_capacity_units_day|current_load_units_day|available_capacity_units_day|status     |
+-------+-----------------------+----------------------+----------------------+----------------------------+-----------+
|Line-1 |WM-7700,WM-8800        |180                   |140                   |40                          |Operational|
|Line-2 |WM-7700,CP-110         |200                   |175                   |25                          |Operational|
|Line-3 |WM-8800,WM-9000        |220                   |198                   |22                          |Degraded   |
|Line-4 |DU-3300,DU-4400        |160                   |95                    |65                          |Operational|
|Line-5 |WM-7700,WM-8800,WM-9000|500                   |120                   |380                         |Operational|
|Line-6 |DU-3300,CP-110         